Connected to 3.10.19 (Python 3.10.19)

In [ ]:
import torch
import torch.nn as nn
import math

#I. Math
#Pytorch: Log Softmax
#https://docs.pytorch.org/docs/stable/generated/torch.nn.LogSoftmax.html
#source code: https://github.com/hkproj/pytorch-transformer

# module 1: InputEmbeddings
class InputEmbeddings(nn.Module): 
    #constructor, def dim of model: d_model, vocab_size(# of words in the vocab )
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        #value 1: embedding length
        self.d_model = d_model
        #value 2: vocab size
        self.vocab_size = vocab_size
        #basically a dictionary layer that maps numbers to same vector ele each time, this vector gets learned by the model
        self.embedding = nn.Embedding(vocab_size, d_model)

    #forward method: use embedding layer via pytorch to
    def forward(self, x):
        #paper 3.4: in the embedding layers, we mb the weights by sqrt of d_model
        #input embeddings are now ready
        return self.embedding(x) * math.sqrt(self.d_model)



In [ ]:
class PositionalEncoding(nn.Module):
    
    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(seq_len, d_model)
       
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)

        #create denominator of formula 1: log space 10,000 / d_model
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        #apply the cos to odd positions,
        #start at 1 and go forward by 2. 1,3,5, etc.
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0) 
        self.register_buffer('pe', pe)

        #the forward method
    def forward(self, x):
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)
        #apply the dropout
        return self.dropout(x)

In [ ]:
class LayerNormalization(nn.Module):
    #constructor. eps: epsilon, a small number you need to give to the model. 10^-6 (0.000001)
    #we need the epsilon bc: consider the denominator:
    #if variance is 0 , then x hat sub j becomes undefined. #Recall: 200/0 = undefined
    #if variance is very small/very close to zero, then x hat sub j becomes very big.
    #Recall: 200/.1 = 2000 very big. then the x hat sub j would get very big.
    #x hat sub j getting very big is undesirable bc the cpu/gpu can only represent
    #numbers up to a certain position and scale, so we don't want
    #very big numbers or very small numbers
    #so for numerical stability, we use epsilon, also to avoid division by 0
    def __init__(self, features, eps: float = 10**-6) -> None:
        super().__init__()
        #save the epsilon
        self.eps = eps
        #alpha (multiplication); bias (additive)
        #nn.Parameter makes the param learnable
        self.alpha = nn.Parameter(torch.ones(features))    #multiplied
        self.bias = nn.Parameter(torch.zeros(features))    #added

    def forward(self, x):
        #-1 bc everything after the batch
        #keepdim bc the mean usually cancels the dimension to which it's applied
        mean = x.mean(dim = -1, keepdim=True)
        std = x.std(dim = -1, keepdim=True)
        return self.alpha * (x - mean) / (std + self.eps) + self.bias

#module / layer 4: feed forward
    #a fully connected layer, that's used in the encoder and the decoder
    #the paper: section 3.3, FFN: Feed Forward Network
    #it is 2 matrices: W1 and W2, mb x with ReLu in between
    #b1 is the added bias
    #can do this in pytorch with linear layer
    #where we define 1st one as W1 + b1, and W2 + b2
    #and in between we apply ReLu (the max)
    #1st one is d model=512 to dff=2048 and back

In [ ]:
class FeedForwardBlock(nn.Module):
    #constructor
    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        #matrix 1
        self.linear_1 = nn.Linear(d_model, d_ff)    #W1 and B1
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)    #W2 and B2

    def forward(self, x):
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))



In [ ]:
class MultiHeadAttentionBlock(nn.Module):
    #h = # of heads
    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        super().__init__()
        #save the values
        self.d_model = d_model
        self.h = h
        assert d_model % h == 0, "d_model is not divisible by h"
        #d_model db h = Dk. (d_model / h) = Dk
        #if we divide d_model by h heads, we get new value: dk
        self.d_k = d_model // h
        #define matrix Wq. d_model x d_model, so output will be seq x d_model
        self.w_q = nn.Linear(d_model, d_model, bias=False)  # Wq
        self.w_k = nn.Linear(d_model, d_model, bias=False)  # Wk
        self.w_v = nn.Linear(d_model, d_model, bias=False)  # Wv
        self.w_o = nn.Linear(d_model, d_model, bias=False)  #Wo
        self.dropout = nn.Dropout(dropout)

    #staticmethod: means you can call fx attention w/o having an instance of this class: MultiHeadAttention
    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        #d_k is the last dim of the query key and the value
        d_k = query.shape[-1]
        # (Batch, h, Seq_Len, d_k) --> (Batch, h, Seq_Len, Seq_Len)
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            attention_scores.masked_fill_(mask == 0, -1e9)
        attention_scores = attention_scores.softmax(dim = -1)   # (Batch, h, 
        if dropout is not None:
                attention_scores = dropout(attention_scores)
        return (attention_scores @ value), attention_scores #return the 
        
    def forward(self, q, k, v, mask):
        #query mb Wq, this produces matrix q prime
        query = self.w_q(q) # (Batch, Seq_Len, d_model) --> (Batch, Seq_Len, d_model), gives us same dim as original Q matrix
        key = self.w_k(k)   # (Batch, Seq_Len, d_model) --> (Batch, Seq_Len, d_model), gives us same dim as original K matrix
        value = self.w_v(v) # (Batch, Seq_Len, d_model) --> (Batch, Seq_Len, d_model), gives us same dim as original V matrix
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1, 2)

        #we want output and attention scores, output of the softmax
        x, self.attention_scores = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)

        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h * self.d_k)

        return self.w_o(x)

In [ ]:
class ResidualConnection(nn.Module):

    def __init__(self, features: int, dropout: float) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        #the skipped connection is between the Add&Norm and the previous layer, so we need the norm
        self.norm = LayerNormalization(features)

#sublayer: the previous layer. we combine x with output of previous layer
#this the def of Add & Norm
    def forward(self, x, sublayer):
        #this deviates from the paper bc other implementations did it this way
        return x + self.dropout(sublayer(self.norm(x)))


In [ ]:
class EncoderBlock(nn.Module):

    def __init__(self, features: int, self_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, dropout: float) -> None:
        super().__init__()
        self.self_attention_block = self_attention_block
        self.feed_forward_block = feed_forward_block
        #moduleList is a way to organize a list modules, in this case 2
        #define our 2 residual connections
        self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(2)])

    #define the source mask: src_mask: the mask we want to apply to the input encoder
    #why need it: we want to hide interaction of padding words with other words

    def forward(self, x, src_mask):
        #1st skip connection is between pe and the 3 part fork before Multi Head Attention (MHA)
        #the x gets sent to 3 part fork Multi Head Attention and 1st Add&Norm (A&N)
        #the 1st x and the 2nd x (lambda x) from MHA output combine at A&N1
        #we define the sublayer lambda x, we first apply the self attention in which we give the query
        #it's called self attention bc: we give the query, key, value is over x our input
        #bc the role of the query, key, value is x itself,
        #so the sentence is watching itself: 1 word of a sentence is interacting with other words of the same sentence
        #this differs from the Decoder bc of cross-attention
        #query coming from Decoder are watching the key and the values coming from the Encoder
        #the self_attention_block (SAB) is calling the def forward(self, q, k, v, mask) fx of the MHA block
        #so we give the SAB, query, key, value and the src_mask
        #this will be combined with the x by using the ResidualConnection
        #this is the residual_connection 1 (RC1)
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, src_mask))
        #this is the residual_connection 2 (RC2): combine output of feed_forward_block with x from RC1
        x = self.residual_connections[1](x, self.feed_forward_block)
        #this defines our Encoder block
        return x
#Now define Encoder objects, can have up to N of them

class Encoder(nn.Module):
    # number of layers = n from ModuleList
    def __init__(self, features: int, layers: nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(features)

    def forward(self, x, mask):
        #apply 1 layer after another
        for layer in self.layers:
            x = layer(x, mask)
            #the output of the previous layer becomes input for next layer
            #finally apply the normalization
        return self.norm(x)
    #0:51 this concludes Encoder portion
    #Skip Connections: each side arrow that bifurcates from MHA and goes to A&N1 along w/ FeedForward (FF) to A&N2 are each 'x'
#Hour 2: Decoder. 0:52:00

#Output Embedding are same as Input Embeddings, the class is the same, we initialize it twice
#PE can use same values for the Decoder as we use for the Encoder
#the 3rd block in the Decoder block is Masked Multi Headed Attention (MMHA) along with Skip Connections
#The Encoder block has 2 sublayers: MHA, FF, 2 SC
#The Decoder block has 3 sublayers: MMHA, MHA, FF, 3 SC

In [ ]:
class EncoderBlock(nn.Module):

    def __init__(self, features: int, self_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, dropout: float) -> None:
        super().__init__()
        self.self_attention_block = self_attention_block
        self.feed_forward_block = feed_forward_block
        #moduleList is a way to organize a list modules, in this case 2
        #define our 2 residual connections
        self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(2)])

    def forward(self, x, src_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, src_mask))
        #this is the residual_connection 2 (RC2): combine output of feed_forward_block with x from RC1
        x = self.residual_connections[1](x, self.feed_forward_block)
        #this defines our Encoder block
        return x

class Encoder(nn.Module):
    # number of layers = n from ModuleList
    def __init__(self, features: int, layers: nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(features)

    def forward(self, x, mask):
        #apply 1 layer after another
        for layer in self.layers:
            x = layer(x, mask)
            #the output of the previous layer becomes input for next layer
            #finally apply the normalization
        return self.norm(x)

In [ ]:
class DecoderBlock(nn.Module):

#the self attention here is the MMHA, not the MHA
#bc the input is used 3x: q, k, v and the same input is used as the q,k,v
#which means each word in the sentence is matched with each other word in the same sentence
#but sublayer 2: MHA, we'll have attention calcuated using query coming from the Decoder
#while the key and values will come from the Encoder
#thus the MHA here is not self-attention, it's called a cross-attention
#it's crossing 2 diff objects together and matching them to calculate the relationship btwn them

    def __init__(self, features: int, self_attention_block: MultiHeadAttentionBlock, cross_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, dropout: float) -> None:
        super().__init__()
        self.features = features
        self.self_attention_block = self_attention_block
        self.cross_attention_block = cross_attention_block
        self.feed_forward_block = feed_forward_block
        #3 residual connections (RC)
        self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(3)])

#x: input of the Decoder
#encoder_output
#src_mask: applied to Encoder, source language
#tgt_mask: target mask applied to Decoder, target language
#we have src_mask and tgt_mask bc we're doing a translation task from source language to target language
    def forward(self, x, encoder_output, src_mask, tgt_mask):
        #call the 1st RC[0]
        #calc self attention
        #x, x, x are q, k, v.
        #q, k, v are same input, but with mask of Decoder, bc this is the self_attention_block of the Decoder
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, tgt_mask))
        #calc cross attention, the 2nd RC[1]. q comes from Decoder, k and v come from Encoder, mask of Encoder
        x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x, encoder_output, encoder_output, src_mask))
        #calc the 3rd RC[2], the FF block
        x = self.residual_connections[2](x, self.feed_forward_block)
        return x
        #this is final component to build a Decoder now, just a matter of # of Decoder blocks, Nx

class Decoder(nn.Module):

    def __init__(self, layers: nn.ModuleList) -> None:
        super().__init__()
        #normalization
        self.layers = layers
        self.norm = LayerNormalization()

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        #apply input to 1 layer, use the output of the previous layer as input of next layer
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
            #we're calling the forward method here: def forward(self, x, encoder_output, src_mask, tgt_mask):
        return self.norm(x)

In [ ]:
class DecoderBlock(nn.Module):

    def __init__(self, features: int, self_attention_block: MultiHeadAttentionBlock, cross_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, dropout: float) -> None:
        super().__init__()
        self.features = features
        self.self_attention_block = self_attention_block
        self.cross_attention_block = cross_attention_block
        self.feed_forward_block = feed_forward_block
        #3 residual connections (RC)
        self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(3)])

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, tgt_mask))
        x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x, encoder_output, encoder_output, src_mask))
        x = self.residual_connections[2](x, self.feed_forward_block)
        return x

class Decoder(nn.Module):
    def __init__(self, layers: nn.ModuleList) -> None:
        super().__init__()
        #normalization
        self.layers = layers
        self.norm = LayerNormalization()

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        #apply input to 1 layer, use the output of the previous layer as input of next layer
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
            #we're calling the forward method here: def forward(self, x, encoder_output, src_mask, tgt_mask):
        return self.norm(x)

In [ ]:
class ProjectionLayer(nn.Module):
    #need d_model
    #need vocab_size
    #this is a linear layer that converts d_model to vocab size
    def __init__(self, d_model: int, vocab_size: int) -> None:
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # (Batch, Seq_Len, d_model) --> (Batch, Seq_Len, Vocab_Size)
        #apply log softmax for numerical stability to the last dim
        return torch.log_softmax(self.proj(x), dim = -1)
        #now we have all components for the Transformer

In [ ]:
class Transformer(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder, src_embed: InputEmbeddings, tgt_embed: InputEmbeddings, src_pos: PositionalEncoding, tgt_pos: PositionalEncoding, projection_layer: ProjectionLayer) -> None:
        super().__init__()
        #save valuesz
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.src_pos = src_pos
        self.tgt_pos = tgt_pos
        self.projection_layer = projection_layer

    #Encoder: we have source language, source mask
    def encode(self, src, src_mask):
        #apply the source embedding
        src = self.src_embed(src)
        #apply the positional encoding
        src = self.src_pos(src)
        #apply the encoder
        return self.encoder(src, src_mask)

    #Decoder: encoder_output: a tensor, src_mask: a tensor
    def decode(self, encoder_output, src_mask, tgt, tgt_mask):
        #apply tgt embedding to tgt sentence
        tgt = self.tgt_embed(tgt)
        #apply pe to tgt sentence
        tgt = self.tgt_pos(tgt)
        #now decode, this is basically the fwd method of this decoder, same order of params
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)

    def project(self, x):
        #apply the projection, take the embedding to the vocabulary size
        return self.projection_layer(x)

In [1]:
class FeedForwardBlock(nn.Module):
    #constructor
    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        #matrix 1
        self.linear_1 = nn.Linear(d_model, d_ff)    #W1 and B1
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)    #W2 and B2

    def forward(self, x):
        #?fuzzy character: input sentence
        #(Batch, Seq_Len, d_model) convert it using Linear 1, into another tensor Batch, Seq_Len, d_ff
        #bc if we apply this Linear d_model into d_ff
        #then we apply linear to convert it back to d_model
        #(Batch, Seq_Len, d_model) --> (Batch, Seq_Len, d_ff) --> (Batch, Seq_Len, d_model)
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))

NameError: name 'nn' is not defined